# Đánh giá kết quả thực nghiệm

## 1. Cấu hình môi trường

In [ ]:
import importlib.util
import json
import math
import os
import pickle
import subprocess
import sys
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        bundled_packages = Path.home() / ".cache" / "codex-runtimes" / "codex-primary-runtime" / "dependencies" / "python"
        if bundled_packages.exists() and str(bundled_packages) not in sys.path:
            sys.path.insert(0, str(bundled_packages))

    if importlib.util.find_spec(import_name) is None:
        print(f"Thiếu thư viện '{import_name}'. Đang cài đặt gói '{pip_name}'...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])


ensure_package("pandas")
ensure_package("matplotlib")

import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False



PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", PROJECT_ROOT / "outputs")).resolve()

ARTIFACT_FILES = {
    "preprocessing_summary_json": os.environ.get("PREPROCESSING_SUMMARY_JSON", "preprocessing_summary.json"),
    "baseline_results_pickle": os.environ.get("BASELINE_RESULTS_PICKLE", "baseline_results.pkl"),
    "baseline_metrics_csv": os.environ.get("BASELINE_METRICS_CSV", "baseline_metrics.csv"),
    "proposed_results_pickle": os.environ.get("PROPOSED_RESULTS_PICKLE", "proposed_results.pkl"),
    "proposed_metrics_csv": os.environ.get("PROPOSED_METRICS_CSV", "proposed_metrics.csv"),
    "evaluation_metrics_csv": os.environ.get("EVALUATION_METRICS_CSV", "evaluation_metrics.csv"),
    "pattern_comparison_csv": os.environ.get("PATTERN_COMPARISON_CSV", "pattern_comparison.csv"),
    "runtime_chart_png": os.environ.get("RUNTIME_CHART_PNG", "runtime_comparison.png"),
    "evaluation_summary_json": os.environ.get("EVALUATION_SUMMARY_JSON", "evaluation_summary.json"),
}


def parse_support_ratios(value, default_ratios):
    if not value:
        return default_ratios
    ratios = []
    for token in value.split(","):
        token = token.strip()
        if not token:
            continue
        if token.endswith("%"):
            ratios.append(float(token[:-1]) / 100)
        else:
            number = float(token)
            ratios.append(number / 100 if number > 1 else number)
    return ratios


MIN_SUPPORT_RATIOS = parse_support_ratios(
    os.environ.get("MIN_SUPPORT_RATIOS", ""),
    default_ratios=[0.05, 0.04, 0.03, 0.02, 0.01],
)
PRICE_THRESHOLD = float(os.environ.get("PRICE_THRESHOLD", "20"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 3000)

print("Môi trường chạy:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Thư mục làm việc:", PROJECT_ROOT)
print("Thư mục lưu kết quả:", OUTPUT_DIR)
print("Danh sách min_support:", [f"{ratio:.0%}" for ratio in MIN_SUPPORT_RATIOS])
print("Ngưỡng avg(price):", PRICE_THRESHOLD)

## 2. Nạp kết quả

In [ ]:
def find_artifact(filename, output_dir=OUTPUT_DIR):
    direct_candidates = [output_dir / filename, PROJECT_ROOT / filename]
    search_roots = [PROJECT_ROOT]
    if IN_COLAB:
        search_roots.append(Path("/content"))

    seen = set()
    for path in direct_candidates:
        resolved = path.resolve()
        if resolved not in seen:
            seen.add(resolved)
            if resolved.exists():
                return resolved

    for root in search_roots:
        if not root.exists():
            continue
        for path in root.glob(f"**/{filename}"):
            resolved = path.resolve()
            if resolved not in seen:
                seen.add(resolved)
                if resolved.exists():
                    return resolved

    return output_dir / filename


summary_path = find_artifact(ARTIFACT_FILES["preprocessing_summary_json"])
baseline_results_path = find_artifact(ARTIFACT_FILES["baseline_results_pickle"])
baseline_metrics_path = find_artifact(ARTIFACT_FILES["baseline_metrics_csv"])
proposed_results_path = find_artifact(ARTIFACT_FILES["proposed_results_pickle"])
proposed_metrics_path = find_artifact(ARTIFACT_FILES["proposed_metrics_csv"])

required_artifacts = [
    summary_path,
    baseline_results_path,
    baseline_metrics_path,
    proposed_results_path,
    proposed_metrics_path,
]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError(
        "Thiếu artifact để đánh giá. Hãy chạy đủ notebook 1, 2 và 3 trước. "
        f"Các file còn thiếu: {missing_artifacts}"
    )

with summary_path.open("r", encoding="utf-8") as f:
    preprocessing_summary = json.load(f)

with baseline_results_path.open("rb") as f:
    baseline_results = pickle.load(f)

with proposed_results_path.open("rb") as f:
    proposed_results = pickle.load(f)

baseline_metrics_raw_df = pd.read_csv(baseline_metrics_path)
proposed_metrics_raw_df = pd.read_csv(proposed_metrics_path)

artifact_df = pd.DataFrame([
    {"Artifact": "Tóm tắt tiền xử lý", "Đường dẫn": str(summary_path)},
    {"Artifact": "Kết quả Baseline", "Đường dẫn": str(baseline_results_path)},
    {"Artifact": "Metrics Baseline", "Đường dẫn": str(baseline_metrics_path)},
    {"Artifact": "Kết quả Proposed", "Đường dẫn": str(proposed_results_path)},
    {"Artifact": "Metrics Proposed", "Đường dẫn": str(proposed_metrics_path)},
])

print("Đã nạp đầy đủ artifact đánh giá:")
display(artifact_df)

print("Tóm tắt dữ liệu tiền xử lý:")
display(pd.DataFrame([
    {"Chỉ tiêu": "Số dòng gốc", "Giá trị": preprocessing_summary.get("raw_rows")},
    {"Chỉ tiêu": "Số dòng sau tiền xử lý", "Giá trị": preprocessing_summary.get("clean_rows")},
    {"Chỉ tiêu": "Số giao dịch", "Giá trị": preprocessing_summary.get("transactions")},
    {"Chỉ tiêu": "Số mặt hàng duy nhất", "Giá trị": preprocessing_summary.get("unique_items")},
    {"Chỉ tiêu": "Cách lấy giá đại diện", "Giá trị": preprocessing_summary.get("price_agg_method", "median")},
]))

## 3. Kiểm tra final patterns của Baseline và Proposed

In [ ]:
def support_label_to_ratio(label):
    if isinstance(label, str) and label.endswith("%"):
        return float(label[:-1]) / 100
    return float(label)


configured_labels = [f"{ratio:.0%}" for ratio in MIN_SUPPORT_RATIOS]
all_labels = []
for label in configured_labels + list(baseline_results.keys()) + list(proposed_results.keys()):
    if label not in all_labels:
        all_labels.append(label)

pattern_comparison_rows = []
difference_examples = []

for label in all_labels:
    baseline_final = baseline_results.get(label, {}).get("final_patterns", {})
    proposed_final = proposed_results.get(label, {}).get("final_patterns", {})

    baseline_itemsets = set(baseline_final)
    proposed_itemsets = set(proposed_final)
    missing_in_proposed = sorted(baseline_itemsets - proposed_itemsets)
    extra_in_proposed = sorted(proposed_itemsets - baseline_itemsets)
    common_itemsets = baseline_itemsets & proposed_itemsets
    support_mismatches = sorted([
        itemset
        for itemset in common_itemsets
        if baseline_final[itemset] != proposed_final[itemset]
    ])

    fully_match = (
        len(missing_in_proposed) == 0
        and len(extra_in_proposed) == 0
        and len(support_mismatches) == 0
    )

    pattern_comparison_rows.append({
        "Min Support": label,
        "Baseline Final Patterns": len(baseline_final),
        "Proposed Final Patterns": len(proposed_final),
        "Khớp hoàn toàn": fully_match,
        "Thiếu trong Proposed": len(missing_in_proposed),
        "Thừa trong Proposed": len(extra_in_proposed),
        "Lệch support": len(support_mismatches),
    })

    for source_name, examples in [
        ("Thiếu trong Proposed", missing_in_proposed[:5]),
        ("Thừa trong Proposed", extra_in_proposed[:5]),
        ("Lệch support", support_mismatches[:5]),
    ]:
        for itemset in examples:
            difference_examples.append({
                "Min Support": label,
                "Loại khác biệt": source_name,
                "Itemset": " | ".join(itemset),
                "Support Baseline": baseline_final.get(itemset),
                "Support Proposed": proposed_final.get(itemset),
            })

pattern_comparison_df = pd.DataFrame(pattern_comparison_rows)
pattern_comparison_df["Support Ratio"] = pattern_comparison_df["Min Support"].apply(support_label_to_ratio)
pattern_comparison_df = pattern_comparison_df.sort_values("Support Ratio", ascending=False).drop(columns=["Support Ratio"]).reset_index(drop=True)

print("Bảng kiểm tra final patterns giữa Baseline và Proposed:")
display(pattern_comparison_df)

if difference_examples:
    print("Các khác biệt mẫu giữa hai phương pháp:")
    display(pd.DataFrame(difference_examples))
else:
    print("Kết luận kiểm tra: Baseline và Proposed khớp hoàn toàn ở tất cả các ngưỡng support.")

## 4. Bảng metrics cuối cho báo cáo

In [ ]:
metrics_rows = []

for label in all_labels:
    if label not in baseline_results or label not in proposed_results:
        continue

    baseline_result = baseline_results[label]
    proposed_result = proposed_results[label]
    baseline_runtime = float(baseline_result["runtime_seconds"])
    proposed_runtime = float(proposed_result["runtime_seconds"])
    final_count = len(baseline_result["final_patterns"])
    speedup = baseline_runtime / proposed_runtime if proposed_runtime > 0 else math.nan
    support_count = baseline_result.get("min_support_count", proposed_result.get("min_support_count"))
    proposed_stats = proposed_result.get("stats", {})

    metrics_rows.append({
        "Min Support": label,
        "Support count tối thiểu": support_count,
        "Thời gian Baseline (giây)": baseline_runtime,
        "Thời gian Proposed (giây)": proposed_runtime,
        "Speedup (Baseline/Proposed)": speedup,
        "Số lượng Final Patterns": final_count,
        "Candidate bị cắt do avg(price)": proposed_stats.get("price_pruned_candidates", 0),
        "Candidate bị loại do support": proposed_stats.get("infrequent_candidates", 0),
    })

evaluation_metrics_df = pd.DataFrame(metrics_rows)
evaluation_metrics_df["Support Ratio"] = evaluation_metrics_df["Min Support"].apply(support_label_to_ratio)
evaluation_metrics_df = evaluation_metrics_df.sort_values("Support Ratio", ascending=False).drop(columns=["Support Ratio"]).reset_index(drop=True)

display_metrics_df = evaluation_metrics_df.copy()
for col in ["Thời gian Baseline (giây)", "Thời gian Proposed (giây)", "Speedup (Baseline/Proposed)"]:
    display_metrics_df[col] = display_metrics_df[col].round(6)

print("Bảng metrics cuối:")
display(display_metrics_df)

report_table_df = display_metrics_df[[
    "Min Support",
    "Thời gian Baseline (giây)",
    "Thời gian Proposed (giây)",
    "Số lượng Final Patterns",
]]

print("Bảng rút gọn đúng định dạng mục 3.4:")
display(report_table_df)

## 5. Biểu đồ so sánh runtime

In [ ]:
chart_path = OUTPUT_DIR / ARTIFACT_FILES["runtime_chart_png"]

x_labels = evaluation_metrics_df["Min Support"].tolist()
baseline_times = evaluation_metrics_df["Thời gian Baseline (giây)"].tolist()
proposed_times = evaluation_metrics_df["Thời gian Proposed (giây)"].tolist()

plt.figure(figsize=(10, 5.8))
plt.plot(x_labels, baseline_times, marker="o", linewidth=2.5, label="Baseline (Post-processing)")
plt.plot(x_labels, proposed_times, marker="s", linewidth=2.5, label="Proposed (In-mining Pruning)")
plt.title("So sánh Runtime giữa Baseline và Proposed theo Min Support")
plt.xlabel("Min Support")
plt.ylabel("Thời gian thực thi (giây)")
plt.grid(True, linestyle="--", alpha=0.35)
plt.legend()
plt.tight_layout()
plt.savefig(chart_path, dpi=160, bbox_inches="tight")
plt.show()

print("Đã lưu biểu đồ runtime tại:", chart_path)

## 6. Lưu bảng đánh giá và tóm tắt

In [ ]:
evaluation_metrics_path = OUTPUT_DIR / ARTIFACT_FILES["evaluation_metrics_csv"]
pattern_comparison_path = OUTPUT_DIR / ARTIFACT_FILES["pattern_comparison_csv"]
evaluation_summary_path = OUTPUT_DIR / ARTIFACT_FILES["evaluation_summary_json"]

display_metrics_df.to_csv(evaluation_metrics_path, index=False, encoding="utf-8-sig")
pattern_comparison_df.to_csv(pattern_comparison_path, index=False, encoding="utf-8-sig")

evaluation_summary = {
    "price_threshold": PRICE_THRESHOLD,
    "all_patterns_match": all_patterns_match,
    "runtime_chart_path": str(chart_path),
    "best_speedup_min_support": str(best_speedup_row["Min Support"]),
    "best_speedup": float(best_speedup_row["Speedup (Baseline/Proposed)"]),
    "metrics": display_metrics_df.to_dict(orient="records"),
    "pattern_comparison": pattern_comparison_df.to_dict(orient="records"),
    "summary_sentences": summary_sentences,
}

with evaluation_summary_path.open("w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, ensure_ascii=False, indent=2)

saved_files_df = pd.DataFrame([
    {"File": str(evaluation_metrics_path), "Mô tả": "Bảng metrics cuối"},
    {"File": str(pattern_comparison_path), "Mô tả": "Bảng kiểm tra final patterns"},
    {"File": str(chart_path), "Mô tả": "Ảnh biểu đồ runtime"},
    {"File": str(evaluation_summary_path), "Mô tả": "Tóm tắt đánh giá dạng JSON"},
])

print("Đã lưu xong các file đánh giá:")
display(saved_files_df)